<a href="https://colab.research.google.com/github/R786P/telco_churn_predictor/blob/main/fooocus_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pygit2==1.15.1
%cd /content
!git clone https://github.com/lllyasviel/Fooocus.git
%cd /content/Fooocus
!python entry_with_update.py --share --always-high-vram


In [ ]:
import pandas as pd

def clean_churn_data(df: pd.DataFrame) -> pd.DataFrame:
    """Cleans raw Telco churn dataset"""
    df = df.copy()

    # 1. TotalCharges has spaces → convert to numeric
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

    # 2. Drop rows with missing critical values
    df = df.dropna(subset=["TotalCharges", "MonthlyCharges"])

    # 3. Encode target variable
    df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

    # 4. Remove non-predictive ID column
    df = df.drop(columns=["customerID"])

    return df.reset_index(drop=True)

In [5]:

import pandas as pd
import joblib
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from data_cleaning import clean_churn_data
from feature_engineering import build_preprocessor

def main():
    # 1. Load data
    df = clean_churn_data(pd.read_csv("data/raw/churn.csv"))
    X = df.drop("Churn", axis=1)
    y = df["Churn"]

    # 2. Load model artifact
    artifact = joblib.load("models/churn_rf_v1.pkl")
    preprocessor = artifact["preprocessor"]
    model = artifact["model"]

    # 3. Transform & Predict
    X_prep = preprocessor.transform(X)
    y_pred = model.predict(X_prep)
    y_prob = model.predict_proba(X_prep)[:, 1]

    # 4. Print Metrics
    print("📊 Classification Report (Churn=1):")
    print(classification_report(y, y_pred, target_names=["No Churn", "Churn"]))

    print("🔲 Confusion Matrix:")
    print(confusion_matrix(y, y_pred))

    print(f"📈 ROC-AUC: {roc_auc_score(y, y_prob):.3f}")

if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'data_cleaning'